# Home Visitation Outcome Predictive

## Problem Framing

**Business question:** Which residents in the visitation workflow are most likely to show supportive near-term home-visit outcomes?

**Operational decision supported:** Support reintegration planning with an early signal about which home environments look more supportive.

**Primary users:** case managers, reintegration staff

**Target summary:** Current predictive label: `label_supportive_home_visit_next_120d`, using future visitation outcomes with favorable, low-concern, cooperative-home signals.

This standardized predictive notebook uses the shared notebook factory so every pipeline follows the same submission structure and evaluation flow.


## Shared Assets And Notebook Bootstrap

Shared references:

* `ml/docs/data-joins.md`
* `ml/docs/feature-catalog.md`
* `ml/docs/phase-3-predictive-pipelines.md`
* `ml/docs/phase-4-modeling-framework.md`
* `ml/docs/phase-b-notebook-standardization.md`


In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "ml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


In [2]:
from ml.src.pipelines.notebook_support import (
    load_evaluation_bundle,
    load_notebook_context,
    summarize_frame,
)

context = load_notebook_context(
    pipeline_name='home_visitation_outcome',
    dataset_name='resident_monthly_features',
    predictive_impl='home_visitation_outcome',
    use_predictive_dataset=True,
)
pipeline_name = context["pipeline_name"]
dataset_name = context["dataset_name"]
config = context["config"]
dataset = context["dataset"]

summarize_frame(dataset)


,resident_id,snapshot_month,safehouse_id,case_status,case_category,sex,initial_risk_level,months_since_admission,age_years_at_snapshot,subcategory_flag_count,...,future_window_complete_60d,future_window_complete_90d,future_window_complete_120d,label_incident_next_30d,label_case_prioritization_next_60d,label_counseling_progress_next_90d,label_education_improvement_next_120d,label_wellbeing_deterioration_next_90d,label_supportive_home_visit_next_120d,label_reintegration_complete_next_90d
0,45,2023-03-01,3,Transferred,Abandoned,F,Medium,1,15.07,2,...,True,True,True,False,False,False,True,False,1,False
1,13,2023-04-01,2,Closed,Surrendered,F,Medium,2,8.89,1,...,True,True,True,False,False,True,True,False,0,False
2,23,2023-04-01,5,Closed,Foundling,F,High,2,9.23,1,...,True,True,True,False,True,True,True,False,0,False
3,45,2023-04-01,3,Transferred,Abandoned,F,Medium,2,15.15,2,...,True,True,True,False,False,False,False,False,1,False
4,29,2023-04-01,8,Transferred,Abandoned,F,Medium,1,12.17,1,...,True,True,True,False,True,True,True,False,0,False
5,13,2023-05-01,2,Closed,Surrendered,F,Medium,3,8.97,1,...,True,True,True,False,False,True,False,False,0,False
6,11,2023-05-01,1,Closed,Surrendered,F,Low,1,17.46,1,...,True,True,True,False,False,False,True,False,0,False
7,54,2023-05-01,7,Transferred,Neglected,F,Medium,1,14.95,0,...,True,True,True,False,True,False,True,False,0,False
8,23,2023-05-01,5,Closed,Foundling,F,High,3,9.31,1,...,True,True,True,False,False,True,True,True,0,False
9,36,2023-05-01,1,Active,Surrendered,F,High,2,12.23,2,...,True,True,True,False,False,True,True,True,0,False


## Target And Leakage Checklist

1. Restate the target in business terms: Current predictive label: `label_supportive_home_visit_next_120d`, using future visitation outcomes with favorable, low-concern, cooperative-home signals.
2. Confirm the snapshot date or split column before running any new model fits.
3. Remove fields that directly encode the future target or post-outcome information.
4. Record any threshold, calibration, or class-balance choice that changes deployment behavior.


## Standard Model Comparison Outputs

Every predictive notebook should read the same evaluation bundle before writing conclusions:

* saved metrics JSON
* Phase 4 holdout comparison table
* Phase 4 cross-validation summary


In [3]:
evaluation = load_evaluation_bundle('home_visitation_outcome')
metrics = evaluation["metrics"]
holdout_comparison = evaluation["holdout_comparison"]
cv_summary = evaluation["cv_summary"]

metrics


{'best_model_name': 'dummy_classifier',
 'train_rows': 612,
 'test_rows': 427,
 'target_col': 'label_supportive_home_visit_next_120d',
 'split_col': 'snapshot_month',
 'selection_metric': 'average_precision',
 'cutoff_date': '2025-04-01',
 'task_type': 'classification',
 'sample_count': 427.0,
 'positive_count': 93.0,
 'positive_rate': 0.21779859484777517,
 'accuracy': 0.7822014051522248,
 'balanced_accuracy': 0.5,
 'precision': 0.0,
 'recall': 0.0,
 'f1': 0.0,
 'roc_auc': 0.5,
 'average_precision': 0.21779859484777517}

In [4]:
summarize_frame(holdout_comparison)


,pipeline_name,model_name,sample_count,positive_count,positive_rate,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,average_precision,mae,rmse,r2
0,home_visitation_outcome,dummy_classifier,427.0,93.0,0.217799,0.782201,0.500000,0.000000,0.000000,0.000000,0.500000,0.217799,NaN,NaN,NaN
1,home_visitation_outcome,logistic_regression,427.0,93.0,0.217799,0.548009,0.513232,0.228261,0.451613,0.303249,0.508403,0.212612,NaN,NaN,NaN
2,home_visitation_outcome,random_forest_classifier,427.0,93.0,0.217799,0.782201,0.500000,0.000000,0.000000,0.000000,0.460241,0.199622,NaN,NaN,NaN


In [5]:
summarize_frame(cv_summary)


,pipeline_name,model_name,fold_mean,fold_std,sample_count_mean,sample_count_std,positive_count_mean,positive_count_std,positive_rate_mean,positive_rate_std,...,roc_auc_std,average_precision_mean,average_precision_std,n_splits,mae_mean,mae_std,rmse_mean,rmse_std,r2_mean,r2_std
0,home_visitation_outcome,random_forest_classifier,3.0,1.581139,122.4,0.547723,27.8,0.447214,0.227122,0.00338,...,0.023543,0.544508,0.064619,5,NaN,NaN,NaN,NaN,NaN,NaN
1,home_visitation_outcome,logistic_regression,3.0,1.581139,122.4,0.547723,27.8,0.447214,0.227122,0.00338,...,0.062307,0.347567,0.053264,5,NaN,NaN,NaN,NaN,NaN,NaN
2,home_visitation_outcome,dummy_classifier,3.0,1.581139,122.4,0.547723,27.8,0.447214,0.227122,0.00338,...,0.000000,0.227122,0.003380,5,NaN,NaN,NaN,NaN,NaN,NaN


## Business Interpretation

Write the final narrative in plain language:

1. What does a high score mean operationally for this pipeline?
2. Which staff action should happen next because of the score?
3. Which users should trust the ranking signal versus wait for more threshold work?
4. Which fairness, bias, or data-quality caveats need to be called out to case managers, reintegration staff?


## Deployment Notes

Recommended shared widgets:

* `insight_summary_card`
* `recommendation_panel`
* `ranked_table_widget`

Deployment checklist:

* Use a reintegration support card and a ranked table in case conference workflows.
* Keep the output framed as planning support rather than a standalone placement decision.

Standard endpoint pattern:

* `POST /ml/predict/home_visitation_outcome`
* `POST /ml/score-batch/home_visitation_outcome`
